# Abalone: exploratory data analysis

Predict shell `Rings` from sex and seven physical measurements. Estimated age in years is `Rings + 1.5`.

**Summary:** the supplied CSV contains 4,177 rows, no missing values or duplicates, and two zero-height records.
The positive-height population contains 4,175 rows. We reserve 835 for the final test and explore only the 3,340 development rows.
Ring counts are right-skewed; sex, size, and weight are associated with rings. These observations motivate a linear baseline and a tree model.

Source: [Kaggle Abalone dataset](https://www.kaggle.com/datasets/rodolfomendes/abalone-dataset), local file `data/abalone.csv`.
Use the project `.venv` kernel, and run cells in order. The dataset is not downloaded by this notebook.

## 1. Setup and scope

Data-quality checks use the supplied file. Target and feature analysis uses the **development split only**;
the test split is reserved for the final evaluation in `modelling.ipynb`.

**Assumptions:** each row is a separate abalone; a random split is appropriate for this coursework dataset.
No animal, farm, or collection-time identifiers are supplied, so independence and real deployment representativeness cannot be established.
Measurements retain the CSV's scale; no physical-unit conversion is inferred.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10})
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 3)

In [ ]:
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start the notebook from the repository root or its notebooks/ directory.")

DATA_PATH = PROJECT_ROOT / "data" / "abalone.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Download the Kaggle Abalone CSV and save it as {DATA_PATH}.")

TARGET = "rings"
CATEGORICAL_COLS = ["sex"]
NUMERICAL_COLS = ["length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight"]
FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS
TEST_SIZE = 0.2
RANDOM_STATE = 42
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

## 2. Load and inspect the source

Column names are normalized to `snake_case` in both notebooks. The file checksum identifies the exact input used.

In [ ]:
raw_df = pd.read_csv(DATA_PATH).rename(columns=lambda name: name.strip().lower().replace(" ", "_"))
missing_columns = set(FEATURE_COLS + [TARGET]) - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

print(f"Source: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"SHA-256: {DATA_SHA256}")
print(f"Rows: {len(raw_df):,}; predictors: {len(FEATURE_COLS)}")
raw_df.head()

In [ ]:
quality_summary = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "missing": raw_df.isna().sum(),
        "unique_values": raw_df.nunique(),
    }
)
print(f"Duplicate rows: {raw_df.duplicated().sum()}")
print(f"Observed sex values: {sorted(raw_df['sex'].dropna().unique())}")
quality_summary

### Height anomalies and the analysis population

Zero height is invalid for the positive-height prediction population. Exclude those rows using a fixed physical rule,
rather than a threshold chosen to improve model scores. The two unusually large positive heights are flagged for inspection;
we retain them because the supplied source does not establish whether they are errors.
Performance will therefore apply to positive-height records, not to all possible incoming measurements.

In [ ]:
invalid_height = raw_df["height"] <= 0
flagged_height = invalid_height | (raw_df["height"] > 0.4)
print(f"Excluded nonpositive heights: {invalid_height.sum()}")
print(f"Flagged positive heights above 0.4: {(raw_df['height'] > 0.4).sum()}")
raw_df.loc[flagged_height, ["sex", "length", "diameter", "height", "whole_weight", TARGET]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].hist(raw_df["height"], bins=40, color="#0072B2", edgecolor="white")
axes[0].set(title="Height, all source rows", xlabel="Height (dataset scale)", ylabel="Abalones")
axes[1].scatter(raw_df["length"], raw_df["height"], alpha=0.25, s=12, color="#0072B2")
axes[1].scatter(
    raw_df.loc[flagged_height, "length"],
    raw_df.loc[flagged_height, "height"],
    s=60,
    marker="x",
    color="#D55E00",
    label="Flagged heights",
)
axes[1].set(title="Inspect height against length", xlabel="Length (dataset scale)", ylabel="Height (dataset scale)")
axes[1].legend()
plt.show()

## 3. Reserve the test set

Use the same fixed seed, filtering rule, and 80/20 split as modelling. This cell does not inspect test targets.

In [ ]:
clean_df = raw_df.loc[~invalid_height].copy()
development_df, test_df = train_test_split(clean_df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Eligible rows: {len(clean_df):,}; development: {len(development_df):,}; held-out test: {len(test_df):,}")
development_df[NUMERICAL_COLS + [TARGET]].describe().T

## 4. Target distribution

Plot integer ring counts with integer-centered bins. Sparse high-ring observations may make errors on older abalones
less stable; this is a hypothesis to check in the final error analysis, not a measured accuracy claim.

In [ ]:
rings = development_df[TARGET]
fig, ax = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
bins = np.arange(rings.min() - 0.5, rings.max() + 1.5, 1)
ax.hist(rings, bins=bins, color="#0072B2", edgecolor="white")
ax.axvline(rings.mean(), color="#D55E00", linestyle="--", label=f"Mean = {rings.mean():.2f}")
ax.set(title=f"Ring distribution: {len(development_df):,} development rows", xlabel="Rings", ylabel="Abalones")
ax.legend()
plt.show()
print(f"Range: {rings.min()}–{rings.max()}; median: {rings.median():.0f}; skewness: {rings.skew():.2f}")

## 5. Sex and ring counts

`I` denotes infant, not a third adult sex. Keep the values nominal: one-hot encoding avoids imposing an artificial order.
Differences between groups are associations and do not establish causality.

In [ ]:
sex_summary = development_df.groupby("sex")[TARGET].agg(["count", "mean", "median"]).reindex(["F", "I", "M"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
sex_summary["count"].plot.bar(ax=axes[0], color="#0072B2", rot=0)
axes[0].set(title="Sex distribution: development rows", xlabel="Sex category", ylabel="Abalones")
sns.boxplot(data=development_df, x="sex", y=TARGET, order=["F", "I", "M"], ax=axes[1], color="#A6CEE3")
axes[1].set(title="Rings by sex: development rows", xlabel="Sex category", ylabel="Rings")
plt.show()
sex_summary

## 6. Numerical distributions and relationships

Inspect measurement ranges and correlations before choosing a model. Strong correlations between measurements
make individual linear coefficients unstable to interpret; they do not by themselves invalidate prediction.

In [ ]:
axes = development_df[NUMERICAL_COLS].hist(bins=35, figsize=(12, 7), color="#0072B2", edgecolor="white")
for ax in axes.flat:
    if ax.has_data():
        ax.set_xlabel("Dataset scale")
        ax.set_ylabel("Abalones")
plt.suptitle("Measurement distributions: development rows", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
correlations = development_df[NUMERICAL_COLS + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
heatmap = ax.imshow(correlations.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
positions = np.arange(len(correlations))
ax.set_xticks(positions, correlations.columns, rotation=90)
ax.set_yticks(positions, correlations.index)
ax.grid(False)
for row in positions:
    for column in positions:
        value = correlations.iloc[row, column]
        ax.text(
            column, row, f"{value:.2f}", ha="center", va="center", color="white" if abs(value) > 0.6 else "#202020", fontsize=9
        )
fig.colorbar(heatmap, ax=ax, label="Pearson correlation")
ax.set_title("Correlations: development rows")
plt.show()
correlations[TARGET].drop(TARGET).sort_values(ascending=False).to_frame("correlation_with_rings")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
for sex, marker, color in [("F", "o", "#0072B2"), ("I", "^", "#009E73"), ("M", "s", "#D55E00")]:
    group = development_df.loc[development_df["sex"] == sex]
    ax.scatter(group["shell_weight"], group[TARGET], label=sex, marker=marker, color=color, alpha=0.3, s=15)
ax.set(title="Shell weight and rings: development rows", xlabel="Shell weight (dataset scale)", ylabel="Rings")
ax.legend(title="Sex category")
plt.show()

## 7. Modelling decisions and limitations

- Use all eight supplied predictors; exclude `rings` from the features.
- Exclude the two zero-height rows. Retain unusual positive values and document their possible influence.
- Encode `sex` with `DictVectorizer` inside a pipeline fitted separately within each cross-validation fold.
- Compare a training-mean baseline, linear regression, and a bounded random forest on development data.
- Use cross-validation for model selection; evaluate the selected model and the mean baseline on the test set once.
- Check prediction bias and error by ring-count group, especially the sparse older group.

This is an exploratory coursework analysis of one dataset. It cannot establish performance on different farms,
populations, or collection periods. No test-set result has been used to choose these modelling decisions.

The same seed-42 holdout was evaluated in an earlier notebook version. In this revision, CV selection uses development rows only,
but that holdout is a reference benchmark rather than a new, untouched confirmatory dataset.